⚽ Proxecto Power BI: Análise de Datos de Fútbol

🎯 Obxectivo

Analizar estatísticas de equipos e xogadores de fútbol, cruzando datos de distintas fontes para obter información valiosa sobre rendemento, condicións meteorolóxicas durante os partidos e outros factores relevantes.

🐍 Script de Scraping en Python

Utilizaremos requests e BeautifulSoup para obter datos meteorolóxicos de partidos desde unha fonte como Time and Date.

In [ ]:
#%pip install fake-useragent

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import time
from datetime import datetime
import random
from fake_useragent import UserAgent

# Configuración
base_url = "https://www.tiempo.com"
current_date = datetime.now().strftime("%Y-%m-%d")
user_agent = UserAgent()

# Diccionario ampliado de ciudades con URLs corregidas
cities = {
    "Madrid": "/madrid.htm",
    "Barcelona": "/barcelona.htm",
    "Valencia": "/valencia.htm",
    "Sevilla": "/sevilla.htm",
    "Zaragoza": "/zaragoza.htm",
    "Málaga": "/malaga.htm",
    "Murcia": "/murcia.htm",
    "Palma": "/palma-de-mallorca.htm",
    "Las Palmas": "/las-palmas-de-gran-canaria.htm",
    "Bilbao": "/bilbao.htm",
    "Alicante": "/alicante-alacant.htm",  
    "Córdoba": "/cordoba.htm",
    "Valladolid": "/valladolid.htm",
    "Vigo": "/vigo.htm",
    "Gijón": "/gijon.htm",
    "Hospitalet de Llobregat": "/gn/3120619.htm",
    "A Coruña": "/coruna-a.htm",  
    "Vitoria-Gasteiz": "/vitoria-gasteiz.htm",
    "Granada": "/granada.htm",
    "Elche": "/elche-elx.htm",
    "Oviedo": "/oviedo.htm",
    "Santa Cruz de Tenerife": "/santa-cruz-de-tenerife.htm",
    "Badalona": "/badalona.htm",
    "Cartagena": "/cartagena.htm",
    "Terrassa": "/terrassa.htm",
    "Jerez de la Frontera": "/jerez-de-la-frontera.htm",
    "Sabadell": "/sabadell.htm",
    "Móstoles": "/mostoles.htm",
    "Alcalá de Henares": "/alcala-de-henares.htm",
    "Pamplona": "/pamplona.htm"
}

# Headers dinámicos
def get_headers():
    return {
        'User-Agent': user_agent.random,
        'Accept-Language': 'es-ES,es;q=0.9',
        'Referer': 'https://www.google.com/',
        'Accept-Encoding': 'gzip, deflate, br'
    }

# Almacenamiento de datos
weather_data = {
    'actual': [],
    'futuro': []
}

def get_weather_data():
    """Función principal para obtener todos los datos meteorológicos"""
    print("=== Iniciando recolección de datos meteorológicos ===")
    
    for city_name, city_path in cities.items():
        print(f"\nProcesando ciudad: {city_name}")
        time.sleep(random.uniform(1, 3))  # Reduce el tiempo de espera entre solicitudes
        
        try:
            url = f"{base_url}{city_path}"
            response = requests.get(url, headers=get_headers())
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')
            
            extract_current_data(soup, city_name)
            extract_forecast_data(soup, city_name)
        
        except Exception as e:
            print(f"Error procesando {city_name}: {str(e)}")
            continue

def extract_current_data(soup, city_name):
    try:
        current = {
            'ciudad': city_name,
            'tipo': 'actual',
            'fecha': current_date,
            'timestamp': datetime.now().isoformat()
        }
        
        temp = soup.select_one('.dato-temperatura.changeUnitT')
        if temp:
            current['temperatura'] = temp.get_text(strip=True)
        
        condition = soup.select_one('.descripcion')
        if condition:
            current['condicion'] = condition.get_text(strip=True)
        
        humidity = soup.select_one('.datos-humedad')
        if humidity:
            current['humedad'] = humidity.get_text(strip=True)
        
        weather_data['actual'].append(current)
        print(f"Datos actuales obtenidos para {city_name}")
        
    except Exception as e:
        print(f"Error extrayendo datos actuales: {str(e)}")

def extract_forecast_data(soup, city_name):
    try:
        forecast_table = soup.select_one('.grid-container-7')
        if not forecast_table:
            print("No se encontró tabla de pronóstico")
            return
            
        rows = forecast_table.select('li')  # Los días están en <li> en lugar de <tr>
        
        for i, row in enumerate(rows[:7]):  # Limitar a los primeros 7 días
            date_element = row.select_one('.fecha')
            temp_element = row.select_one('.temperatura')
            condition_element = row.select_one('.descripcion')
            
            if all([date_element, temp_element, condition_element]):
                forecast = {
                    'ciudad': city_name,
                    'tipo': 'pronostico',
                    'fecha': date_element.get_text(strip=True),
                    'temperatura': temp_element.get_text(strip=True),
                    'condicion': condition_element.get_text(strip=True),
                    'timestamp': datetime.now().isoformat()
                }
                weather_data['futuro'].append(forecast)
        
        print(f"Pronóstico futuro obtenido para {city_name} (7 días)")
        
    except Exception as e:
        print(f"Error extrayendo pronóstico futuro: {str(e)}")

def save_data():
    try:
        all_data = weather_data['actual'] + weather_data['futuro']
        df = pd.DataFrame(all_data)
        df['fecha'] = pd.to_datetime(df['fecha'], dayfirst=True, errors='coerce')
        df.sort_values(by=['ciudad', 'fecha'], inplace=True)
        
        csv_file = f"datos_tiempo_completos_{current_date}.csv"
        df.to_csv(csv_file, index=False, encoding='utf-8-sig')
        
        json_file = f"datos_tiempo_completos_{current_date}.json"
        with open(json_file, 'w', encoding='utf-8') as f:
            json.dump(all_data, f, ensure_ascii=False, indent=2)
        
        print(f"\n✅ Datos guardados en {csv_file} y {json_file}")
        print("\nResumen de datos recolectados:")
        print(f"- Actuales: {len(weather_data['actual'])} registros")
        print(f"- Futuros: {len(weather_data['futuro'])} registros")
        
        return df
        
    except Exception as e:
        print(f"Error guardando datos: {str(e)}")
        return None

if __name__ == "__main__":
    start_time = time.time()
    
    get_weather_data()
    weather_df = save_data()
    
    end_time = time.time()
    print(f"\nTiempo total de ejecución: {round(end_time - start_time, 2)} segundos")
    
    if weather_df is not None:
        print("\nPrimeras filas del DataFrame:")
        print(weather_df.head())


Este script recolle as temperaturas das principais cidades de España e gárdaas en formato JSON e CSV para a súa posterior análise.

🐼 Procesamento de Datos con Pandas en Power BI

In [ ]:
# Cargar datos de clima desde el archivo CSV generado previamente
clima_df = pd.read_csv("datos_tiempo_completos_2025-05-12.csv")

# Limpiar los datos
clima_df = clima_df.dropna()  # Eliminar filas con valores nulos
clima_df["ciudad"] = clima_df["ciudad"].str.strip().str.title()  # Normalizar nombres de ciudades

# Mostrar el DataFrame limpio
clima_df


Este script limpa os datos meteorolóxicos e prepáraos para a súa integración con outras fontes de datos en Power BI.

🔗 Integración con Spark-HDFS

Para integrar datos almacenados en HDFS, podemos utilizar PySpark:

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("FootballData").getOrCreate()

# Ler datos de partidos desde HDFS
df = spark.read.parquet("hdfs://localhost:9000/datos_futbol/partidos.parquet")
df.show()


Estes datos poden incluír información detallada sobre partidos, como resultados, estatísticas de xogadores, etc.

In [ ]:
# Combinar datos reales del clima con los equipos de cada ciudad
datos_combinados = pd.merge(
    asistencia_df,
    clima_df[clima_df["tipo"] == "actual"],  # Filtrar solo datos reales
    left_on="Cidade",
    right_on="ciudad",
    how="inner"
)

# Seleccionar columnas relevantes
datos_combinados = datos_combinados[["Equipo", "Estadio", "Cidade", "temperatura", "condicion", "Asistencia", "Capacidade"]]

# Mostrar los datos combinados
datos_combinados


In [ ]:
# Guardar el DataFrame combinado en un archivo CSV
datos_combinados.to_csv("datos_combinados_powerbi.csv", index=False, encoding="utf-8-sig")

print("✅ Archivo 'datos_combinados_powerbi.csv' guardado exitosamente.")

<h1>Graficas Power Bi<h1>

In [ ]:
##Graficar la temperatura media por ciudad
# Importar librerías necesarias
import pandas as pd
import matplotlib.pyplot as plt

# Cargar los datos
clima_df = pd.read_csv('C:/Users/constantin.madalin.i/Tareas-y-Ejercicios-de-Clase2/Tareas-y-Ejercicios-de-Clase/SBD/Clase de Profesor Victor/Proyecto Futbol/datos_combinados_powerbi.csv')

# Limpiar temperatura
clima_df['temperatura'] = clima_df['temperatura'].str.replace('°', '', regex=False).astype(float)

# Agrupar por ciudad y calcular temperatura media (puedes cambiar a mediana si prefieres)
media_temp = clima_df.groupby('Cidade')['temperatura'].mean().sort_values()

# Crear gráfico de línea
plt.figure(figsize=(10, 6))
plt.plot(media_temp.index, media_temp.values, marker='o', linestyle='-', color='b')

# Estética
plt.xticks(rotation=90)
plt.ylabel("Temperatura media (°C)")
plt.title("Temperatura media por cidade")
plt.grid(True)
plt.tight_layout()
plt.show()
plt.show()


In [ ]:
##Gráfico de barras comparando Asistencia y Capacidad por Equipo
# Importar librerías necesarias

import pandas as pd
import matplotlib.pyplot as plt

# Cargar datos
df = pd.read_csv('C:/Users/constantin.madalin.i/Tareas-y-Ejercicios-de-Clase2/Tareas-y-Ejercicios-de-Clase/SBD/Clase de Profesor Victor/Proyecto Futbol/datos_combinados_powerbi.csv')

# Convertir columnas a numéricas
df['asistencia'] = pd.to_numeric(df['Asistencia'], errors='coerce')
df['capacidade'] = pd.to_numeric(df['Capacidade'], errors='coerce')

# Agrupar por Equipo y Cidade
agrupado = df.groupby(['Equipo', 'Cidade'], as_index=False).agg({
    'asistencia': 'mean',
    'capacidade': 'mean'
})

# Crear gráfico de barras comparativo
plt.figure(figsize=(14, 6))
x = range(len(agrupado))
plt.bar(x, agrupado['capacidade'], width=0.4, label='Capacidade', align='center')
plt.bar([i + 0.4 for i in x], agrupado['asistencia'], width=0.4, label='Asistencia', align='center')

# Etiquetas
plt.xticks([i + 0.2 for i in x], agrupado['Equipo'], rotation=90)
plt.ylabel("Número de personas")
plt.title("Asistencia vs Capacidad por Equipo")
plt.legend()
plt.tight_layout()
plt.show()


<h1>Scraping Futbol<h1>

In [ ]:
from selenium import webdriver
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
from datetime import datetime, timedelta

# CONFIGURACIÓN
NUM_DIAS = 7  # Días a partir de hoy
URL = "https://www.flashscore.es/futbol/espana/laliga-ea-sports/"

# CONFIGURAR FIREFOX EN MODO HEADLESS
options = Options()
options.headless = True
driver = webdriver.Firefox(options=options)

# ABRIR LA PÁGINA
print("Cargando página...")
driver.get(URL)

# ESPERAR CARGA DE PARTIDOS
try:
    WebDriverWait(driver, 20).until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".event__match"))
    )
except Exception as e:
    print(f"No se cargaron los partidos: {e}")
    driver.quit()
    exit()

# EXTRAER PARTIDOS
partidos = []
hoy = datetime.now()
fecha_limite = hoy + timedelta(days=NUM_DIAS)

bloques = driver.find_elements(By.CSS_SELECTOR, ".event__match--scheduled")

for partido in bloques:
    try:
        fecha_str = partido.get_attribute("data-day")  # '20250515'
        hora_elem = partido.find_element(By.CSS_SELECTOR, ".event__time")
        equipo_local = partido.find_element(By.CSS_SELECTOR, ".event__participant--home").text.strip()
        equipo_visitante = partido.find_element(By.CSS_SELECTOR, ".event__participant--away").text.strip()
        hora = hora_elem.text.strip()

        if not fecha_str or not hora:
            continue

        fecha = datetime.strptime(fecha_str, "%Y%m%d")
        hora_dt = datetime.strptime(hora, "%H:%M").time()
        fecha_hora = datetime.combine(fecha, hora_dt)

        if hoy <= fecha_hora <= fecha_limite:
            partidos.append({
                "Fecha y Hora": fecha_hora.strftime("%Y-%m-%d %H:%M"),
                "Equipo Local": equipo_local,
                "Equipo Visitante": equipo_visitante
            })

    except Exception as e:
        print(f"⚠️ Error procesando un partido: {e}")
        continue

driver.quit()

# GUARDAR Y MOSTRAR
df = pd.DataFrame(partidos)
df.to_csv("laliga_flashscore_semana.csv", index=False, encoding="utf-8")
df.to_json("laliga_flashscore_semana.json", orient="records", force_ascii=False, indent=2)

print(f"\n✅ Se encontraron {len(df)} partidos en los próximos {NUM_DIAS} días:")
print(df)
